# Churn Predictor
This notebook aims to study the data gathered from a Bank to predict customer churn.

The game plan is the following:
1. **Exploratory Data Analysis (EDA)**
2. Data Preprocessing
    + Data Unsampling using SMOTE
    + Principal Component Analysis Of One Hot Encoded Data
3. Model Selection and Evaluation
    + Cross Validation
    + Model Evaluation
    + Model Evaluation On Original Data (Before Upsampling)
4. Results

In [ ]:
import pandas as pd

data = pd.read_csv("BankChurners.csv")

print("Columns: ", data.columns)
print("\n")
print("First: \n ", data.head(3))


Columns:  Index(['CLIENTNUM', 'Attrition_Flag', 'Customer_Age', 'Gender',
       'Dependent_count', 'Education_Level', 'Marital_Status',
       'Income_Category', 'Card_Category', 'Months_on_book',
       'Total_Relationship_Count', 'Months_Inactive_12_mon',
       'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal',
       'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt',
       'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio',
       'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1',
       'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2'],
      dtype='object')


First: 
     CLIENTNUM     Attrition_Flag  Customer_Age Gender  Dependent_count  \
0  768805383  Existing Customer            45      M                3   
1  818770008  Existing Customer            49      F             

# Exploratory Data Analysis — Bank Customer Churn

### 1. Dataset Overview
- Number of rows and columns
- Data types (numerical vs categorical)
- Missing values per column
- Unique values for categorical features

### 2. Target Variable Analysis
- Distribution of `Attrition_Flag`
- Churn rate (% of attrited customers)
- Class imbalance assessment

### 3. Customer Demographics vs Churn
Analyze how churn varies with:
- Customer_Age
- Gender
- Education_Level
- Marital_Status
- Income_Category
- Dependent_count

Visuals:
- Bar plots (churn rate by category)
- Boxplots (age vs churn)

### 4. Relationship & Account Features
Explore:
- Months_on_book
- Total_Relationship_Count
- Card_Category
- Credit_Limit

Questions:
- Do customers with fewer products churn more?
- Does tenure reduce churn?

### 5. Activity & Engagement Indicators
Focus on:
- Months_Inactive_12_mon
- Contacts_Count_12_mon

Hypothesis:
- Higher inactivity → higher churn

### 6. Transaction Behavior Analysis
Key variables:
- Total_Trans_Amt
- Total_Trans_Ct
- Avg_Utilization_Ratio
- Total_Amt_Chng_Q4_Q1
- Total_Ct_Chng_Q4_Q1

Goals:
- Identify behavioral patterns of churned customers
- Detect early warning signals

### 7. Correlation Analysis
- Correlation matrix for numerical features
- Identify strongest predictors of churn
- Detect multicollinearity

### 8. Outliers & Distributions
- Distribution plots for numerical variables
- Outlier detection (boxplots, IQR)

### 9. Feature Importance (EDA-level)
- Compare feature distributions between churned vs non-churned
- Preliminary ranking of influential variables

### 10. EDA Summary & Modeling Hypotheses
Summarize:
- Key churn drivers
- Features to prioritize
- Features to drop or transform


## Data
The data combines **demographic information**, **account characteristics**, and **transaction behavior**, making it suitable for supervised classification tasks.

### Target Variable

* **Attrition_Flag**

  * *Existing Customer*: Customer is still active
  * *Attrited Customer*: Customer has left the bank
    This is the binary target variable to be predicted.

### Customer Identification

* **CLIENTNUM**
  Unique customer identifier.
  > This column should be **excluded from modeling**, as it has no predictive value.

### Demographic Features

* **Customer_Age** – Age of the customer
* **Gender** – Customer gender (M / F)
* **Dependent_count** – Number of dependents
* **Education_Level** – Education level of the customer
* **Marital_Status** – Marital status
* **Income_Category** – Annual income range

These variables describe the socio-demographic profile of the customer.

### Account & Relationship Features

* **Card_Category** – Type of credit card (e.g., Blue, Silver, Gold)
* **Months_on_book** – Length of relationship with the bank (in months)
* **Total_Relationship_Count** – Number of products held by the customer
* **Credit_Limit** – Credit limit assigned to the customer

These features capture the depth and longevity of the customer–bank relationship.

### Activity & Engagement Features

* **Months_Inactive_12_mon** – Number of inactive months in the last year
* **Contacts_Count_12_mon** – Number of contacts with the bank in the last year

These variables are strong indicators of customer engagement and potential churn risk.


### Transaction Behavior Features

* **Total_Revolving_Bal** – Total revolving balance
* **Avg_Open_To_Buy** – Available credit
* **Total_Trans_Amt** – Total transaction amount (last 12 months)
* **Total_Trans_Ct** – Total transaction count (last 12 months)
* **Total_Amt_Chng_Q4_Q1** – Change in transaction amount (Q4 vs Q1)
* **Total_Ct_Chng_Q4_Q1** – Change in transaction count (Q4 vs Q1)
* **Avg_Utilization_Ratio** – Average credit utilization ratio

These features describe **customer spending behavior and usage intensity**, which are typically among the most predictive variables for churn.


### Precomputed Model Columns

* **Naive_Bayes_Classifier_Attrition_Flag_…_1**
* **Naive_Bayes_Classifier_Attrition_Flag_…_2**

These columns contain probabilities generated by a pre-trained Naive Bayes model and **must be removed** before training new models to avoid **data leakage**. 

Data leakage happens when the model has access to information that:
- Would not be available at prediction time, or
- Is directly derived from the target variable


In [ ]:
df = data.copy()

df = df.drop(columns=[
    'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1',
       'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2'
])